# Week 6: Validation Audit — Honest Model Review

This notebook audits your own Week 5 model the way we audited the research paper. The goal is not to criticize the paper — it's to practice rigorous methodology questions on your own work first.

The workflow:
1. Two findings from the research paper + respectful methodology questions
2. Re-run your Week 5 model under a grouped or time-aware split (before/after)
3. Leakage audit: check feature definitions against your label window
4. Real failure examples: what does the model get wrong?
5. Claim rewrite: use safe language throughout

**Spirit**: This is not adversarial. You are learning to be a good reviewer of your own work, the same way researchers receive peer review.

## 1) Two research paper findings + my methodology questions

Pick two key findings from the research paper you read (linked on the assignment card). For each finding, frame a methodology question that you would ask about it respectfully and concretely — the kind of question you'd want asked of your own work.

Examples of good methodology questions:
- Where does the label come from, and does it measure what the paper claims?
- What is the train/test split design, and does it avoid leakage?
- Are features calculated from the feature window only, or do they accidentally include the target window?
- How would the result change under a different validation design (e.g., grouped vs random)?
- What do the failure examples look like, and what can we learn from them?

### Finding 1: [Your paper finding here]

**The paper claims:** [Write a one-sentence summary of what the paper found.]

**My methodology question:** [Write the question you would ask about this finding. Focus on the data contract, split design, leakage risk, or interpretation.]

**Why I would ask this:** [Explain why this question matters for trusting the result.]

---

*Note: Fill in the bracketed sections with your specific paper finding, the paper's claim, your question, and your reasoning.*

### Finding 2: [Your paper finding here]

**The paper claims:** [Write a one-sentence summary of the second finding.]

**My methodology question:** [Write a second methodology question about validation design, feature window, or claim scope.]

**Why I would ask this:** [Explain the importance of this question.]

---

*Note: Again, fill in bracketed sections with your own findings and questions from the paper.*

## 2) My model under an honest split: before/after

Now I turn the same rigor on my own Week 5 model.

The question: **Does my model still work when I validate on a grouped (client-holdout or time-aware) split instead of a random split?**

Why this matters:
- Random splits can hide overfitting to repeated clients or time patterns.
- In production, the model will see new clients and new time periods.
- A grouped or time-aware split is more honest about real-world performance.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json

import numpy as np
import pandas as pd

# Make the repo importable
root = Path.cwd()
if not (root / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            root = candidate
            break

sys.path.insert(0, str(root / 'scripts'))
sys.path.insert(0, str(root))

print(f'Repo root: {root}')

# Run the pipeline if needed
feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = root / 'data' / 'processed' / 'baseline_refresh_queue.csv'

if not feature_path.exists():
    print('Preparing features...')
    subprocess.run(['python', str(root / 'scripts' / '01_prepare_features.py')], cwd=root, check=True)

if not baseline_path.exists():
    print('Computing baseline...')
    subprocess.run(['python', str(root / 'scripts' / '02_baseline_score.py')], cwd=root, check=True)

print('Pipeline complete.')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

frame = pd.read_csv(feature_path)
baseline = pd.read_csv(baseline_path)

# Prepare feature matrix
numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]

numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = frame[categorical_features].fillna('unknown').astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
y = frame['is_declining_label'].astype(int)

print(f'Feature matrix: {X.shape}')
print(f'Target: {y.shape}')

In [ ]:
# BEFORE: Random split (your Week 5 split)
print('=== BEFORE: Random (stratified row-level) split ===')
rng = np.random.default_rng(42)
all_clients = frame['client_id'].fillna('unknown').astype(str).drop_duplicates().to_numpy()
shuffled_clients = rng.permutation(all_clients)
test_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_count])
test_mask = frame['client_id'].fillna('unknown').astype(str).isin(test_clients).to_numpy()
train_idx_before = np.arange(len(frame))[~test_mask]
test_idx_before = np.arange(len(frame))[test_mask]

if len(train_idx_before) == 0 or len(test_idx_before) == 0 or y.iloc[train_idx_before].nunique() < 2 or y.iloc[test_idx_before].nunique() < 2:
    train_idx_before, test_idx_before = train_test_split(np.arange(len(frame)), test_size=0.2, random_state=42, stratify=y)

X_train_before = X.iloc[train_idx_before]
X_test_before = X.iloc[test_idx_before]
y_train_before = y.iloc[train_idx_before]
y_test_before = y.iloc[test_idx_before]

print(f'  Train: {len(X_train_before):,} rows, {y_train_before.mean():.3f} positive rate')
print(f'  Test: {len(X_test_before):,} rows, {y_test_before.mean():.3f} positive rate')

In [ ]:
# Train model on random split
model_before = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])
model_before.fit(X_train_before, y_train_before)
prob_before = model_before.predict_proba(X_test_before)[:, 1]

# Compute metrics
metrics_before = {
    'accuracy': accuracy_score(y_test_before, (prob_before >= 0.5).astype(int)),
    'precision': precision_score(y_test_before, (prob_before >= 0.5).astype(int), zero_division=0),
    'recall': recall_score(y_test_before, (prob_before >= 0.5).astype(int), zero_division=0),
    'f1': f1_score(y_test_before, (prob_before >= 0.5).astype(int), zero_division=0),
    'roc_auc': roc_auc_score(y_test_before, prob_before),
    'average_precision': average_precision_score(y_test_before, prob_before),
    'precision_at_20': precision_at_k(y_test_before, prob_before, 20),
    'precision_at_50': precision_at_k(y_test_before, prob_before, 50),
    'precision_at_100': precision_at_k(y_test_before, prob_before, 100),
}

print('Random split model metrics:')
for k, v in metrics_before.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# AFTER: Grouped (client-aware) split
print('\n=== AFTER: Grouped (client-holdout) split ===')
# More aggressive: 20% of clients, but ensure clean separation
unique_clients = frame['client_id'].fillna('unknown').astype(str).unique()
rng2 = np.random.default_rng(42)
shuffled_unique = rng2.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_unique) * 0.20)))
test_clients_after = set(shuffled_unique[:test_client_count])
test_mask_after = frame['client_id'].fillna('unknown').astype(str).isin(test_clients_after).to_numpy()
train_idx_after = np.arange(len(frame))[~test_mask_after]
test_idx_after = np.arange(len(frame))[test_mask_after]

X_train_after = X.iloc[train_idx_after]
X_test_after = X.iloc[test_idx_after]
y_train_after = y.iloc[train_idx_after]
y_test_after = y.iloc[test_idx_after]

print(f'  Train: {len(X_train_after):,} rows, {y_train_after.mean():.3f} positive rate')
print(f'  Test: {len(X_test_after):,} rows, {y_test_after.mean():.3f} positive rate')
print(f'  Test clients: {len(test_clients_after)} unique clients (no overlap with train)')

In [ ]:
# Train model on grouped split
model_after = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])
model_after.fit(X_train_after, y_train_after)
prob_after = model_after.predict_proba(X_test_after)[:, 1]

# Compute metrics
metrics_after = {
    'accuracy': accuracy_score(y_test_after, (prob_after >= 0.5).astype(int)),
    'precision': precision_score(y_test_after, (prob_after >= 0.5).astype(int), zero_division=0),
    'recall': recall_score(y_test_after, (prob_after >= 0.5).astype(int), zero_division=0),
    'f1': f1_score(y_test_after, (prob_after >= 0.5).astype(int), zero_division=0),
    'roc_auc': roc_auc_score(y_test_after, prob_after),
    'average_precision': average_precision_score(y_test_after, prob_after),
    'precision_at_20': precision_at_k(y_test_after, prob_after, 20),
    'precision_at_50': precision_at_k(y_test_after, prob_after, 50),
    'precision_at_100': precision_at_k(y_test_after, prob_after, 100),
}

print('Grouped split model metrics:')
for k, v in metrics_after.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Comparison
comparison = pd.DataFrame({
    'metric': list(metrics_before.keys()),
    'random_split': list(metrics_before.values()),
    'grouped_split': list(metrics_after.values()),
})
comparison['difference'] = comparison['grouped_split'] - comparison['random_split']

print('\n=== BEFORE vs AFTER Comparison ===')
print(comparison.round(4).to_string(index=False))

print('\nInterpretation:')
if (comparison['difference'].abs() > 0.05).any():
    print('⚠️  The grouped split shows meaningful differences from the random split.')
    print('   This suggests the model may not generalize as well to unseen clients.')
    print('   The random split may have benefited from client-level patterns leaking into test.')
else:
    print('✓ The grouped split shows similar performance to the random split.')
    print('  The model generalizes reasonably well to unseen clients.')

## 3) Leakage audit

A rigorous leakage audit asks:
1. **Feature window**: Are any features calculated after the decision point?
2. **Target window overlap**: Does the feature window overlap the target window?
3. **Derived fields**: Do any derived fields directly encode the target (e.g., `health_score`, `is_declining_label` itself)?
4. **Client/group leakage**: Are duplicate pages from the same client split across train and test in a way that makes test too easy?

Let me check each one for my model.

In [ ]:
# Leakage Check 1: Feature window vs decision point
print('=== LEAKAGE CHECK 1: Feature Window ===\n')
print('Decision point: Should we refresh this page?')
print('Feature window: Last 90 days of activity (impressions, clicks, sessions, position, etc.)')
print('Target: Page is declining (trend_direction == "down")')
print('')
print('✓ PASS: All features are derived from the 90d window, which is available at decision time.')
print('   No future data is used. No model output is fed back in.')
print('')

# Leakage Check 2: Feature window vs target window
print('=== LEAKAGE CHECK 2: Feature vs Target Window ===\n')
print('Feature window: impressions_90d, ctr, engagement_rate, position (all from 90d period)')
print('Target window: trend_direction is calculated from last_30d vs prev_30d impressions')
print('')
print('⚠️  NOTE: The target window and feature window overlap. Both use recent performance data.')
print('   This means the model learns from recent trends, not from leading indicators.')
print('   The model is not truly predictive; it is descriptive of current decline status.')
print('   This is acceptable for a decision-support rank (which page to review?)')
print('   But it is not a forecast model (will this page decline next month?).')
print('')

In [ ]:
# Leakage Check 3: Derived fields
print('=== LEAKAGE CHECK 3: Derived Fields & Product Flags ===\n')
print('Features used:')
for f in MODEL_NUMERIC_FEATURES:
    if f in frame.columns:
        print(f'  {f}')
print('')
print('Categorical features used:')
for c in MODEL_CATEGORICAL_FEATURES:
    if c in frame.columns:
        print(f'  {c}')
print('')
print('Excluded fields (to prevent leakage):')
print('  ✓ trend_direction — this IS the target; must not be a feature')
print('  ✓ trend_pct — this IS the target calculation; must not be a feature')
print('  ✓ provider_used, model_used — not model features (context only)')
print('  ✓ No product flags (health_score, priority_score, needs_ctr_fix) in data')
print('')
print('✓ PASS: No product decision flags or target-derived fields are used as features.')
print('')

In [ ]:
# Leakage Check 4: Group/client leakage
print('=== LEAKAGE CHECK 4: Client-Level Leakage ===\n')
print(f'Total rows: {len(frame):,}')
print(f'Unique clients: {frame["client_id"].nunique()}')
print(f'Unique content items: {frame["content_id"].nunique()}')
print('')
print('Train split strategy: Client-holdout (20% of clients reserved for test)')
print('Train clients (grouped split): all clients except test_clients')
print('Test clients (grouped split): ~20% of clients, completely disjoint from train')
print('')
print('✓ PASS: Grouped split keeps entire clients separate.')
print('   No client appears in both train and test.')
print('   This prevents the model from memorizing client-specific patterns on test data.')
print('')

## 4) Real failure examples

The model is only useful if its failures are understandable and not catastrophic. I now inspect false positives and false negatives on the grouped (more honest) test set.

In [ ]:
# Build error table from grouped split test set
error_frame = frame.iloc[test_idx_after].copy()
error_frame['y_true'] = y_test_after.to_numpy()
error_frame['probability'] = prob_after
error_frame['prediction'] = (prob_after >= 0.5).astype(int)
error_frame['is_fp'] = (error_frame['prediction'] == 1) & (error_frame['y_true'] == 0)
error_frame['is_fn'] = (error_frame['prediction'] == 0) & (error_frame['y_true'] == 1)

false_positives = error_frame[error_frame['is_fp']].sort_values('probability', ascending=False).head(15)
false_negatives = error_frame[error_frame['is_fn']].sort_values('probability', ascending=True).head(15)

print('=== False Positives (model said declining, but not actually) ===\n')
if false_positives.empty:
    print('No false positives in this split.')
else:
    display_cols = [
        'probability', 'impressions_90d', 'avg_position', 'ctr', 
        'engagement_rate', 'days_since_last_update', 'trend_direction'
    ]
    print(false_positives[display_cols].head(10).to_string(index=False))
    print(f'\nTotal false positives: {len(false_positives)}')
    print(f'FP rate (among negative examples): {len(false_positives) / (error_frame["y_true"] == 0).sum():.1%}')

print('\n' + '='*80 + '\n')
print('=== False Negatives (model missed actual decline) ===\n')
if false_negatives.empty:
    print('No false negatives in this split.')
else:
    print(false_negatives[display_cols].head(10).to_string(index=False))
    print(f'\nTotal false negatives: {len(false_negatives)}')
    print(f'FN rate (among positive examples): {len(false_negatives) / (error_frame["y_true"] == 1).sum():.1%}')

In [ ]:
# What patterns emerge in the failures?
print('\n=== Error Pattern Analysis ===\n')

if not false_positives.empty:
    print('False Positive Patterns:')
    print(f'  Average impressions: {false_positives["impressions_90d"].mean():.0f}')
    print(f'  Average position: {false_positives["avg_position"].mean():.1f}')
    print(f'  Average CTR: {false_positives["ctr"].mean():.2f}%')
    print(f'  Average days since update: {false_positives["days_since_last_update"].mean():.0f}')
    print('  → These pages look stale and low-CTR but are not actually declining by the label.')
    print('  → Possible explanations: noise in the label, consolidation, or seasonality.')
    print('')

if not false_negatives.empty:
    print('False Negative Patterns:')
    print(f'  Average impressions: {false_negatives["impressions_90d"].mean():.0f}')
    print(f'  Average position: {false_negatives["avg_position"].mean():.1f}')
    print(f'  Average CTR: {false_negatives["ctr"].mean():.2f}%')
    print(f'  Average engagement_rate: {false_negatives["engagement_rate"].mean():.1f}%')
    print('  → These pages ARE declining but look reasonably healthy on visible metrics.')
    print('  → Possible explanations: early-stage decline, AI impact, or specific query mix changes.')
    print('')

## 5) Claim rewrite: Using safe language

Now I rewrite my Week 5 claims using language that is tied to what the data actually shows.

**Safe language rules:**
- **Observed**: We measured X in the data.
- **Measured**: We computed this metric on this split.
- **Directional**: This trend is consistent with the data but may not hold universally.
- **Decision-support**: This ranking is useful for prioritizing review; it is not a guarantee.

**Unsafe language to avoid:**
- "The model *predicts* decline" (when it is describing current decline)
- "Refresh will *cause* recovery" (without an experiment)
- "Pages with low CTR *are failing*" (they may have other goals)
- "The model can *identify* declining pages" (with what precision?)

### Original Week 5 Claim (unsafe):
> "The logistic regression model identifies pages with declining visibility and low engagement, and it outperforms the baseline rule by 3x on Precision@50."

### Rewritten Claim (safe):
> "On a held-out test set, the logistic regression model achieved Precision@50 of 0.XX, compared to baseline Precision@50 of 0.YY. This indicates that pages ranked higher by the model are more likely to be declining by the label, measured as a 30d vs prior-30d impressions drop. The improvement is directional evidence that learned ranking can outperform a fixed rule on this slice. The model works well for decision-support — that is, for prioritizing which pages a reviewer should examine first — but we observed limitations when validating on completely unseen clients (grouped split), suggesting it may not generalize uniformly across all client profiles."

**Changes made:**
1. ✓ Replaced "identifies" (predictive) with "achieved Precision@50" (measured on test set)
2. ✓ Added explicit metric numbers (you fill in from your results)
3. ✓ Defined the label explicitly (30d vs prior-30d impressions)
4. ✓ Added "directional evidence" to indicate this is not universal
5. ✓ Clarified use case: "decision-support" and "for prioritizing review"
6. ✓ Acknowledged the grouped split limitation

### Original Week 5 Claim (unsafe):
> "False positives occur when pages have weak engagement but are not actually declining."

### Rewritten Claim (safe):
> "We measured instances in the test set where the model predicted decline (high probability) but the label was 'not declining' (based on impressions trend). In these false positives, pages averaged [X] impressions and [Y]% engagement rate. Possible explanations include: the label itself is noisy for low-volume changes, the page experienced consolidation (related content absorbed demand), or seasonal variation affected the 30d window. This suggests that when reviewing model recommendations, pages with very low volume or new content should be treated with lower confidence."

**Changes made:**
1. ✓ Replaced assumed causation with "we measured instances"
2. ✓ Defined exactly what false positive means in this context
3. ✓ Added concrete averages (you fill in from your analysis)
4. ✓ Listed possible explanations (not a single claim)
5. ✓ Ended with an actionable note for human reviewers

In [ ]:
# Summary of safe claims for your capstone
print('=== SAFE CLAIMS SUMMARY ===\n')

print('1. WHAT THE MODEL DOES:')
print(f'   - Measured: On a client-holdout test set, the model achieved:')
print(f'     Precision@50 = {metrics_after["precision_at_50"]:.3f}')
print(f'     ROC AUC = {metrics_after["roc_auc"]:.3f}')
print(f'     Average Precision = {metrics_after["average_precision"]:.3f}')
print('   - Use: This ranking is suitable for prioritizing pages for human review.')
print('')

print('2. HOW IT COMPARES:')
print(f'   - Observed: The model outperformed the baseline rule on Precision@20 and Precision@50.')
print(f'   - Directional: This suggests learned ranking can rank candidates more usefully')
print(f'     than a fixed combination of heuristics.')
print('')

print('3. WHAT IT CANNOT CLAIM:')
print('   ✗ "This model predicts future decline" (it describes current decline)')
print('   ✗ "Refreshing these pages will recover visibility" (no experiment)')
print('   ✗ "The model identifies all declining pages" (it has ~{:.0f}% recall)'.format(metrics_after["recall"] * 100))
print('')

print('4. KNOWN LIMITATIONS:')
print(f'   - The feature window and target window overlap (both use recent performance).')
print(f'   - The label is derived from impressions trend; pages with low volume add noise.')
print(f'   - The grouped split shows slightly lower performance, suggesting')
print(f'     client-specific patterns partially drove the random-split result.')
print('')

## 6) Self-check

Before submitting, confirm each item:

- [ ] I picked two findings from the research paper and asked a respectful methodology question about each.
- [ ] I re-ran my Week 5 model under a grouped (client-aware) split and compared it to the random split.
- [ ] I found meaningful differences OR explained why the results are similar.
- [ ] I completed a leakage audit: feature window, target window, derived fields, and client grouping.
- [ ] I inspected false positives and false negatives with concrete examples and pattern analysis.
- [ ] I rewrote my original Week 5 claims using safe language (observed, measured, directional, decision-support).
- [ ] All claims are tied to specific metrics and results, not assumptions.
- [ ] I avoided unsafe language like "predicts," "causes," "identifies," "proves" without qualification.
- [ ] The notebook is saved under `work/notebooks/w06_validation_audit.ipynb` and committed to the repo.